# Persian News Classification with Multinomial Naive Bayes

This notebook implements a multinomial Naive Bayes classifier from scratch,
including Persian text cleaning, evaluation, frequent-word charts, and optional
word clouds. Run it from the `notebooks/` directory after installing
`requirements.txt`.


In [ ]:
# Core imports
import math
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
from hazm import stopwords_list


class DataPreProcessor:
    def __init__(self):
        """
        Simple data preprocessor for Persian text.
        Stores the dataset and defines a basic stopword list.
        """
        self.data = None
        self.stopwords = set(stopwords_list())
        self.stopwords.update({"ها", "های","ی","ای" ,"ترین","تر"})

    def read_data(self, path):
        """
        Read a CSV file from the given path and store it in self.data.

        Parameters
        ----------
        path : str
            Path to the CSV file.

        Returns
        -------
        pandas.DataFrame
            Loaded dataset.
        """
        self.data = pd.read_csv(path)
        return self.data

    def split_data(self, X, y, test_size=0.2):
        """
        Split data into train and test sets while preserving
        label distribution (stratified split).

        Parameters
        ----------
        X : array-like
            Input features (e.g. text column).
        y : array-like
            Labels.
        test_size : float, optional
            Fraction of data to use for the test set, by default 0.2.

        Returns
        -------
        X_train, X_test, y_train, y_test
        """
        return train_test_split(
            X,
            y,
            test_size=test_size,
            stratify=y,
            random_state=42
        )

    def clean_text(self, text):
        """
        Clean a single text sample:
        - Convert to string
        - Remove URLs and emails
        - Remove numbers (Persian, Arabic, English)
        - Remove English letters
        - Keep only Persian letters and spaces
        - Remove one-character tokens
        - Collapse long repeated characters
        - Normalize spaces
        - Remove simple Persian stopwords

        Parameters
        ----------
        text : str
            Raw text.

        Returns
        -------
        str
            Cleaned text.
        """
        # Ensure input is string
        text = str(text)

        # Normalize some Persian characters
        text = text.replace("ي", "ی").replace("ك", "ک")

        # Remove URLs
        text = re.sub(r"http\S+|www\.\S+", " ", text)

        # Remove emails
        text = re.sub(r"\S+@\S+", " ", text)

        # Remove numbers (Persian + Arabic + English)
        text = re.sub(r"[0-9۰-۹٠-٩]", " ", text)

        # Remove English letters
        text = re.sub(r"[a-zA-Z]", " ", text)

        # Keep only Persian letters and whitespace (remove punctuation/symbols)
        text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

        # Remove single-character tokens (e.g., isolated letters)
        text = re.sub(r"\b\w{1}\b", " ", text)

        # Collapse long repeated characters (e.g., "عاااالی" -> "عاالی")
        text = re.sub(r"(.)\1{2,}", r"\1", text)

        # Normalize multiple spaces
        text = re.sub(r"\s+", " ", text).strip()

        # Remove stopwords
        words = text.split()
        words = [w for w in words if w not in self.stopwords]
        text = " ".join(words)

        return text


In [ ]:
import math
from collections import defaultdict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score


class NaiveBayesClassifier:
    def __init__(self, X, y):
        """
        Naive Bayes text classifier (multinomial model).

        Parameters
        ----------
        X : iterable of str
            Cleaned text samples (sentences/documents).
        y : iterable
            Class labels for each sample.
        """
        # Store training data
        self.X = list(X)
        self.y = list(y)

        # Basic stats
        self.n_samples = len(self.y)
        self.classes = sorted(set(self.y))

        # Pair (sentence, label) for easier iteration
        self.data = list(zip(self.X, self.y))

        # Count how many samples each class has (for priors)
        self.class_counts = defaultdict(int)
        for label in self.y:
            self.class_counts[label] += 1

        # These will be filled by count_word_per_class()
        self.word_counts = None                # dict[class][word] = count
        self.total_words_per_class = {}        # dict[class] = total #words in that class
        self.vocab = set()                     # set of all words
        self.V = 0                             # vocabulary size

        # Build word statistics immediately
        self.count_word_per_class()

    def count_word_per_class(self, output_path=None):
        """
        Count how many times each word appears in each class.

        Also:
        - print number of unique words for each class
        - save top 200 most frequent words per class to a text file

        Parameters
        ----------
        output_path : str, optional
            File path where top 200 words per class will be saved.
        """
        word_counts = defaultdict(lambda: defaultdict(int))

        # Count word occurrences per class
        for sentence, label in self.data:
            words = str(sentence).split()
            for w in words:
                word_counts[label][w] += 1

        self.word_counts = word_counts

        # Build vocabulary and total word counts per class
        self.vocab = set()
        self.total_words_per_class = {}
        for cls, wc in self.word_counts.items():
            self.vocab.update(wc.keys())
            self.total_words_per_class[cls] = sum(wc.values())
        self.V = len(self.vocab)

        # Print number of unique words per class
        for cls, wc in self.word_counts.items():
            print(f"Class {cls}: {len(wc)} unique words")

        if output_path is not None:
            with open(output_path, "w", encoding="utf-8") as f:
                for cls, wc in self.word_counts.items():
                    f.write(f"Class {cls}\n")
                    top_words = sorted(
                        wc.items(), key=lambda x: x[1], reverse=True
                    )[:200]
                    for word, count in top_words:
                        f.write(f"{word}\t{count}\n")
                    f.write("\n")

        return self.word_counts

    def calculate_word_log_prob_per_class(self, word, _class, alpha=1.0):
        """
        Compute log P(word | class) using multinomial Naive Bayes
        with Laplace (add-alpha) smoothing.

        Parameters
        ----------
        word : str
            Word for which probability is computed.
        _class : hashable
            Target class label.
        alpha : float, optional
            Smoothing parameter (default 1.0).

        Returns
        -------
        float
            Log probability of the word in the given class.
        """
        # Total number of words in this class
        total_words_in_class = self.total_words_per_class[_class]

        # Number of times this word appears in this class
        class_word_counts = self.word_counts[_class]
        count_wc = class_word_counts.get(word, 0)

        # Laplace smoothing
        numerator = count_wc + alpha
        denominator = total_words_in_class + alpha * self.V

        prob = numerator / denominator
        return math.log(prob)

    def calculate_log_prior(self, _class):
        """
        Compute log prior probability of a class:
        log P(class) = log( count(class) / total_samples ).

        Parameters
        ----------
        _class : hashable
            Class label.

        Returns
        -------
        float
            Log prior probability of the class.
        """
        count_c = self.class_counts[_class]
        prior = count_c / self.n_samples
        return math.log(prior)

    def predict(self, sentence, verbose=False):
        """
        Predict class label for a single sentence.

        The method:
        - splits the sentence into words
        - for each class, sums log prior + sum of log P(word|class)
        - prints the log-scores for all classes
        - prints and returns the class with maximum score

        Parameters
        ----------
        sentence : str
            Cleaned text sample.

        Returns
        -------
        predicted_class
        """
        # Ignore tokens unseen during training. Treating each unseen token as
        # if it had class-specific smoothed probability can bias the prediction.
        words = [word for word in str(sentence).split() if word in self.vocab]

        log_scores = {}

        # Compute posterior log-score for each class
        for cls in self.classes:
            score = self.calculate_log_prior(cls)
            for w in words:
                score += self.calculate_word_log_prob_per_class(w, cls)
            log_scores[cls] = score

        predicted_class = max(log_scores, key=log_scores.get)
        if verbose:
            print("Log-scores per class:", log_scores)
            print("Predicted class:", predicted_class)
        return predicted_class

    def evaluate(self, y_true, y_pred):
        """
        Evaluate model performance using confusion matrix
        and classification report.

        Parameters
        ----------
        y_true : array-like
            True class labels.
        y_pred : array-like
            Predicted class labels.

        Returns
        -------
        tuple
            (confusion_matrix, classification_report_str, accuracy)
        """
        cm = confusion_matrix(y_true, y_pred)
        report = classification_report(y_true, y_pred, digits=4)
        acc = accuracy_score(y_true, y_pred)

        print("Confusion matrix:\n", cm)
        print("\nClassification report:\n", report)
        print(f"Accuracy: {acc:.4f}")

        return cm, report, acc


In [ ]:
# 1) Read data
dp = DataPreProcessor()
df = dp.read_data("../data/persian_news.csv")

# 2) Split raw text before fitting any text statistics.
X = df["Text"]
y = df["Topic"]
X_train, X_test, y_train, y_test = dp.split_data(X, y, test_size=0.25)

# 3) Apply deterministic text cleaning to each partition.
X_train_clean = X_train.apply(dp.clean_text)
X_test_clean = X_test.apply(dp.clean_text)

# 4) Train and predict without printing one score dictionary per document.
nb = NaiveBayesClassifier(X_train_clean, y_train)
y_pred = [nb.predict(sentence) for sentence in X_test_clean]

# 5) Evaluate.
nb.evaluate(y_test, y_pred)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm, _, _ = nb.evaluate(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=nb.classes,
            yticklabels=nb.classes)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap")
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=nb.classes, zero_division=0
)

plt.figure(figsize=(8,5))
plt.bar(nb.classes, f1, color=["#5DADE2", "#48C9B0", "#F5B041"])
plt.title("F1 Score per Class")
plt.xlabel("Class")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.show()


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display
from matplotlib import font_manager


font_candidates = [
    Path("../assets/Vazirmatn-Regular.ttf"),
    Path("/Library/Fonts/Arial Unicode.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Unicode.ttf"),
]
FONT_PATH = next((path for path in font_candidates if path.exists()), None)

if FONT_PATH is not None:
    font_manager.fontManager.addfont(FONT_PATH)
    mpl.rcParams["font.family"] = font_manager.FontProperties(
        fname=FONT_PATH
    ).get_name()
else:
    print(
        "Optional Persian font not found. Add assets/Vazirmatn-Regular.ttf "
        "for best chart and word-cloud rendering."
    )

for cls in nb.classes:
    top_words = sorted(
        nb.word_counts[cls].items(), key=lambda item: item[1], reverse=True
    )[:10]
    words = [get_display(arabic_reshaper.reshape(word)) for word, _ in top_words]
    counts = [count for _, count in top_words]

    plt.figure(figsize=(10, 5))
    plt.bar(words, counts, color="#4C84D9")
    plt.title(f"Top 10 Words in Class '{cls}'", fontsize=16)
    plt.xticks(rotation=45, ha="right", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
from wordcloud import WordCloud

if FONT_PATH is None:
    print("Skipping word clouds because no Persian-capable font was found.")
else:
    for cls in nb.classes:
        top_items = sorted(
            nb.word_counts[cls].items(),
            key=lambda item: item[1],
            reverse=True,
        )[:200]

        frequencies = {}
        for word, count in top_items:
            display_word = get_display(arabic_reshaper.reshape(word))
            frequencies[display_word] = frequencies.get(display_word, 0) + count

        cloud = WordCloud(
            width=1200,
            height=600,
            background_color="white",
            font_path=str(FONT_PATH),
        ).generate_from_frequencies(frequencies)

        plt.figure(figsize=(14, 7))
        plt.imshow(cloud, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"Word Cloud for Class: {cls}", fontsize=18)
        plt.show()
